### PyTorch AlexNet Exercises

Welcome to the PyTorch AlexNet exercise template notebook.

There are several questions in this notebook and it's your goal to answer them by writing Python and PyTorch code.






In [ ]:
!wget http://cs231n.stanford.edu/tiny-imagenet-200.zip
!unzip tiny-imagenet-200.zip

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import numpy as np
import time
from tqdm import tqdm

# Define the AlexNet architecture
class AlexNet(nn.Module):
    def __init__(self, num_classes=200):  # Tiny ImageNet has 200 classes
        super(AlexNet, self).__init__()
        # Define the layers of AlexNet
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=11, stride=4, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(64, 192, kernel_size=5, stride=1, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(192, 384, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(),
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Linear(4096, num_classes)
        )

    def forward(self, x):
        # Define the forward pass
        x = self.features(x)
        x = self.classifier(x)
        return x

# Hyperparameters
learning_rates = [0.1, 0.001, 0.0001]
batch_sizes = [16, 32, 64]

# Define transforms for the input data
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize images to match AlexNet input
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Load the Tiny ImageNet dataset
# Note: You'll need to download the dataset and set the correct path.
train_dataset = datasets.ImageFolder('tiny-imagenet-200/train', transform=transform)
val_dataset = datasets.ImageFolder('tiny-imagenet-200/val', transform=transform)

# Loop over hyperparameters
results = {}
for lr in learning_rates:
    for batch_size in batch_sizes:
        # Data loaders
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

        # Initialize the network
        net = AlexNet(num_classes=200).cuda()

        # Loss and optimizer
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.SGD(net.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)

        # Train the network
        print(f'Training with learning rate: {lr}, batch size: {batch_size}')
        num_epochs = 10

        for epoch in range(1, num_epochs + 1):

            # --- TRAINING LOOP ---
            train_loss = 0.0
            net.train()

            # Avvolgiamo il train_loader con tqdm
            with tqdm(train_loader, unit="batch", leave=False) as tepoch:
                tepoch.set_description(f"Epoch {epoch}/{num_epochs} [Train]")

                for inputs, labels in tepoch:
                    inputs, labels = inputs.cuda(), labels.cuda()

                    optimizer.zero_grad()
                    outputs = net(inputs)
                    loss = criterion(outputs, labels)
                    loss.backward()
                    optimizer.step()

                    train_loss += loss.item() * inputs.size(0)

                    tepoch.set_postfix(loss=loss.item())

            # --- VALIDATION LOOP ---
            val_loss = 0.0
            correct = 0
            total = 0
            net.eval()

            with torch.no_grad(), tqdm(val_loader, unit="batch", leave=False) as tepoch:
                tepoch.set_description(f"Epoch {epoch}/{num_epochs} [Val]")

                for inputs, labels in tepoch:
                    inputs, labels = inputs.cuda(), labels.cuda()
                    outputs = net(inputs)
                    loss = criterion(outputs, labels)

                    val_loss += loss.item() * inputs.size(0)

                    _, predicted = torch.max(outputs.data, 1)
                    total += labels.size(0)
                    correct += (predicted == labels).sum().item()

            train_loss /= len(train_loader.dataset)
            val_loss /= len(val_loader.dataset)
            val_accuracy = 100 * correct / total

            print(f'Epoch [{epoch}/{num_epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.2f}%')

        # You can save the model state if you want to keep it
        # torch.save(net.state_dict(), f'alexnet_lr{lr}_bs{batch_size}.pth')
        results[(lr, batch_size)] = val_accuracy

# To analyze the effect of weight decay, you can vary the weight_decay parameter
# in the SGD optimizer and repeat the training and evaluation process.

for params, accuracy in results.items():
    lr, batch_size = params
    print(f'Learning Rate: {lr}, Batch Size: {batch_size}, Validation Accuracy: {accuracy:.2f}%')

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Training with learning rate: 0.1, batch size: 16


Epoch [1/10], Train Loss: 5.3135, Val Loss: 5.9244, Val Accuracy: 0.00%


Epoch [2/10], Train Loss: 5.3135, Val Loss: 5.4693, Val Accuracy: 0.00%


Epoch [3/10], Train Loss: 5.3139, Val Loss: 5.5353, Val Accuracy: 0.00%


...

Epoch [4/10], Train Loss: 5.2981, Val Loss: 5.2982, Val Accuracy: 0.00%


Epoch [5/10], Train Loss: 5.2980, Val Loss: 5.2987, Val Accuracy: 0.00%


Epoch [6/10], Train Loss: 5.2978, Val Loss: 5.3001, Val Accuracy: 0.00%


Epoch [7/10], Train Loss: 5.2976, Val Loss: 5.3023, Val Accuracy: 0.00%


Epoch 8/10 [Train]:   2%|▏         | 73/3125 [00:06<03:37, 14.04batch/s, loss=5.3]